# 01: Colab inference with the frozen bundle `v2-A7` (GPU-first, parity-safe)

**Sequence:**

1. Mount Drive.
2. Clone the runtime repo.
3. Install the requirements.
4. Detect the GPU.
5. Copy the **complete** bundle from Drive to local disk.
6. Copy the competition files.
7. Verify the manifest.
8. Verify `CONFIG_HASH`.
9. Verify model, aggregator and calibration.
10. Force the frozen `bundle/code`.
11. Run the bundle self-test.
12. Select a validated backend.
13. Run inference.
14. Validate the submission.
15. Write `run_report.json`.
16. Write `submission.csv`.
17. Copy the results to a **new** Drive folder.

**Code authority.** The GitHub repo holds orchestration only (`casmi_runtime`). Inference runs the frozen
`bundle/code/casmi_infer`, bound to `CONFIG_HASH`, and the report proves it.

**GPU policy.** A GPU backend is used only if the frozen bundle implements one, declares it parity-validated,
*and* it passes the bundle self-test. Bundle v2-A7 ships none, so a GPU runtime runs the best CPU backend that
passes the self-test (numba, else numpy), and `fallback_reason` says why. LightGBM prediction of the frozen text
boosters is CPU either way. Nothing is trained or refit, and no training data is needed. Submitting is manual.

In [ ]:
# 1. mount Drive + configuration
from google.colab import drive
drive.mount('/content/drive')

from datetime import datetime, timezone
from pathlib import Path

GITHUB_USER = 'YOUR_GITHUB_USER'          # <- your GitHub account
REPO_NAME = 'enveda-casmi-runtime'
REPO_BRANCH = 'main'
DEVICE_PREFERENCE = 'gpu'                 # prefer a GPU backend IF validated
REQUIRE_GPU = False                       # production: a validated CPU backend is fine; True only for benchmarking
EXPECTED_CONFIG_HASH = None               # optional: paste the CONFIG_HASH printed by YOUR local export to pin it
FORCE_RECOPY_BUNDLE = False

DRIVE_ROOT = Path('/content/drive/MyDrive/EnvedaCASMI')
DRIVE_BUNDLE, DRIVE_COMPETITION, DRIVE_RESULTS = DRIVE_ROOT / 'bundle', DRIVE_ROOT / 'competition', DRIVE_ROOT / 'results'
RUNTIME_DIR = Path('/content/casmi_runtime')               # LOCAL disk -- heavy work never reads the mounted Drive
INPUT_ROOT = RUNTIME_DIR / 'input'
LOCAL_BUNDLE, LOCAL_COMPETITION = INPUT_ROOT / 'bundle', INPUT_ROOT / 'competition'
OUTPUT_ROOT = RUNTIME_DIR / 'output'                        # outside INPUT_ROOT
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPO_DIR = Path('/content') / REPO_NAME
COMPETITION_FILES = ('test.parquet', 'sample_submission.csv')

assert GITHUB_USER != 'YOUR_GITHUB_USER', 'set GITHUB_USER'
for p in [DRIVE_BUNDLE / 'manifest.json', DRIVE_BUNDLE / 'config.json', *(DRIVE_COMPETITION / f for f in COMPETITION_FILES)]:
    assert p.exists(), f'missing on Drive: {p} (run 00_colab_setup.ipynb / see COLAB_RUN_GUIDE.md)'
print('run id', RUN_ID)

In [ ]:
# 2. clone / update the GitHub runtime repo (Colab Secret GITHUB_TOKEN for a private repo; never printed or persisted)
import base64, subprocess


def _github_token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        return None


def _git(args):
    token = _github_token()
    auth = []
    if token:
        basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        auth = ['-c', f'http.extraHeader=Authorization: Basic {basic}']   # lives only for this one git call
    r = subprocess.run(['git', *auth, *args], capture_output=True, text=True)
    del token, auth
    if r.returncode != 0:
        raise RuntimeError(f'git {args[0]} failed (exit {r.returncode}):\n{r.stderr[-2000:]}')
    return r.stdout


if (REPO_DIR / '.git').exists():
    _git(['-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    _git(['clone', '--depth', '1', '--branch', REPO_BRANCH, f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git', str(REPO_DIR)])
REPO_COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('runtime repo', REPO_DIR, '@', REPO_COMMIT)

In [ ]:
# 3. runtime requirements (lightgbm pinned for the frozen boosters; everything else only if missing)
import sys
from importlib.metadata import version
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab.txt')], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
r.check_returncode()
assert version('lightgbm') == '4.7.0', 'lightgbm must be 4.7.0: Runtime -> Restart session, then re-run from cell 1'

In [ ]:
# 4. accelerator detection (hardware). The orchestration package comes from the clone; it imports NO inference code.
import json
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from casmi_runtime.accelerator import detect_accelerator
from casmi_runtime.config import RuntimeConfig
from casmi_runtime.frozen import load_frozen_code
from casmi_runtime.inference import check_bundle_identity, run_frozen_inference
from casmi_runtime.results import save_results_immutable
ACCEL = detect_accelerator()
print(json.dumps({k: ACCEL[k] for k in ('device', 'gpu_name', 'gpu_memory_total_mb', 'cuda_driver_version', 'cupy_available', 'numba_available',
                                        'numba_cuda_available', 'lightgbm_version', 'cpu_count', 'ram_gb')}, indent=1, default=str))

In [ ]:
# 5. COMPLETE bundle Drive -> local disk (never file-by-file; reused only when a completion marker matches the manifest)
import hashlib, shutil, time


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


assert str(LOCAL_BUNDLE).startswith('/content/') and not str(LOCAL_BUNDLE).startswith('/content/drive')
MARKER = RUNTIME_DIR / 'bundle_copy_complete.json'
DRIVE_MANIFEST_SHA = sha256(DRIVE_BUNDLE / 'manifest.json')
if (not FORCE_RECOPY_BUNDLE and LOCAL_BUNDLE.exists() and MARKER.exists()
        and json.loads(MARKER.read_text()).get('manifest_sha256') == DRIVE_MANIFEST_SHA):
    print('reusing the local bundle copy (same manifest sha256)')
else:
    MARKER.unlink(missing_ok=True)
    if LOCAL_BUNDLE.exists():
        shutil.rmtree(LOCAL_BUNDLE)
    LOCAL_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    shutil.copytree(DRIVE_BUNDLE, LOCAL_BUNDLE, ignore=shutil.ignore_patterns('__pycache__', '.ipynb_checkpoints'))
    MARKER.write_text(json.dumps({'manifest_sha256': DRIVE_MANIFEST_SHA, 'seconds': time.time() - t0}))
    print(f'copied the whole bundle in {time.time() - t0:.0f}s')
man = json.loads((LOCAL_BUNDLE / 'manifest.json').read_text(encoding='utf-8'))
missing = [f for f in man['files'] if not (LOCAL_BUNDLE / f).exists()]
assert not missing, f'{len(missing)} manifest files missing locally, e.g. {missing[:5]} -- re-copy the whole bundle to Drive'
print(f"{len(man['files'])} manifest files present | {sum(p.stat().st_size for p in LOCAL_BUNDLE.rglob('*') if p.is_file()) / 1e9:.2f} GB")

In [ ]:
# 6. competition files -> local disk (nothing else under INPUT_ROOT: inputs are discovered by schema)
LOCAL_COMPETITION.mkdir(parents=True, exist_ok=True)
for name in COMPETITION_FILES:
    shutil.copy2(DRIVE_COMPETITION / name, LOCAL_COMPETITION / name)
stray = [p for p in INPUT_ROOT.rglob('*') if p.is_file() and not (p.is_relative_to(LOCAL_BUNDLE) or p.parent == LOCAL_COMPETITION)]
extra = sorted(p.name for p in LOCAL_COMPETITION.iterdir() if p.name not in COMPETITION_FILES)
assert not stray and not extra, f'unexpected files under {INPUT_ROOT}: {stray[:5]} {extra[:5]}'
print({n: f'{(LOCAL_COMPETITION / n).stat().st_size / 1e6:.1f} MB' for n in COMPETITION_FILES})

In [ ]:
# 7-10. FROZEN code first; manifest sha256s + CONFIG_HASH; aggregator + frozen calibration contract; frozen identity
FROZEN = load_frozen_code(LOCAL_BUNDLE)                      # bundle/code first on sys.path; every module's origin proven
assert Path(FROZEN.source).is_relative_to((LOCAL_BUNDLE / 'code').resolve()), FROZEN.source
MANIFEST, CONFIG, problems = FROZEN.validation.verify_bundle(LOCAL_BUNDLE)          # 7 + 8
assert not problems, f'bundle verification FAILED -- inference blocked: {problems[:10]}'
CONTRACT = FROZEN.validation.verify_aggregator_contract(CONFIG, LOCAL_BUNDLE)       # 9 (raises on missing / mismatched calibration)
MODEL_INFO = json.loads((LOCAL_BUNDLE / 'models' / 'model_info.json').read_text(encoding='utf-8'))
CFG = RuntimeConfig(runtime='colab', input_root=INPUT_ROOT, work_dir=OUTPUT_ROOT / RUN_ID, device_preference=DEVICE_PREFERENCE,
                    require_gpu=REQUIRE_GPU, expected_config_hash=EXPECTED_CONFIG_HASH,
                    extra={'github_commit': REPO_COMMIT, 'drive_manifest_sha256': DRIVE_MANIFEST_SHA})
check_bundle_identity(CONFIG, MANIFEST, MODEL_INFO, CFG)    # model / aggregator / v2-A7 / FROZEN / optional pinned CONFIG_HASH
print(f"bundle {MANIFEST['bundle_version']} | CONFIG_HASH {CONFIG['CONFIG_HASH']} | model {CONFIG['model_id']} | aggregator {CONTRACT['name']} "
      f"| T {CONTRACT['temperature']} | calibration sha256 {CONTRACT['calibration_sha256'][:16]} | code {FROZEN.source}")

In [ ]:
# 11. bundle self-test in a FRESH process on the frozen code (format-2 fixture: spectrum + molecule level; numpy + numba).
# 12. The inference run repeats the self-test IN-PROCESS for every candidate backend and activates the first that
#     passes (validated GPU -> numba -> numpy -> STOP); see actual_backend / fallback_reason in the report.
import os
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join(p for p in env.get('PYTHONPATH', '').split(os.pathsep) if p and not Path(p).resolve().is_relative_to(REPO_DIR))
SELFTEST_PASSED = False
p = subprocess.Popen([sys.executable, str(REPO_DIR / 'scripts' / 'v63_bundle_check.py'), '--bundle', str(LOCAL_BUNDLE),
                      '--expect', CONFIG['aggregator_name']], env=env, cwd='/content', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
if p.wait() != 0:
    raise RuntimeError('bundle self-test FAILED -- inference blocked')
SELFTEST_PASSED = True

In [ ]:
# 13-16. inference -> validation -> run_report.json -> submission.csv (orchestration calling ONLY the frozen code)
assert SELFTEST_PASSED, 'bundle self-test did not pass -- inference is blocked'
REPORT = run_frozen_inference(CFG)

In [ ]:
# inspect: frozen code ran, identity unchanged, validations passed, which backend and why
import pandas as pd
assert Path(REPORT['casmi_infer_source']).resolve().is_relative_to((LOCAL_BUNDLE / 'code').resolve()), REPORT['casmi_infer_source']
assert REPORT['CONFIG_HASH'] == CONFIG['CONFIG_HASH'] and REPORT['model_id'] == CONFIG['model_id'] and REPORT['aggregator'] == CONFIG['aggregator_name']
assert REPORT['self_test_status'] == 'PASS' and REPORT['submission_validation_status'] == 'PASS'
keys = ('runtime_platform', 'python_version', 'gpu_available', 'gpu_name', 'requested_device', 'actual_backend', 'gpu_used', 'numba_available',
        'numpy_fallback_used', 'fallback_reason', 'bundle_version', 'CONFIG_HASH', 'model_id', 'aggregator', 'calibration_hash',
        'casmi_infer_source', 'self_test_status', 'n_spectra', 'n_molecules', 'candidate_pair_count', 'runtime_seconds', 'peak_ram',
        'peak_gpu_memory', 'fallback_counts', 'exception_count', 'submission_rows', 'submission_validation_status', 'submission_sha256')
print(json.dumps({k: REPORT.get(k) for k in keys}, indent=1, default=str))
sub = pd.read_csv(CFG.work_dir / 'submission.csv')
print(sub.shape)
print(sub.head(3).assign(smiles=lambda d: d['smiles'].str.slice(0, 120)).to_string(index=False))

In [ ]:
# 17. results -> a NEW Drive folder results/<UTC timestamp>_<CONFIG_HASH>/ (never overwritten; sha256-verified copies)
DEST = save_results_immutable(CFG.work_dir, DRIVE_RESULTS, CONFIG['CONFIG_HASH'], provenance={
    'runtime_platform': 'colab', 'github_repo': f'{GITHUB_USER}/{REPO_NAME}', 'github_commit': REPO_COMMIT,
    'drive_bundle_manifest_sha256': DRIVE_MANIFEST_SHA, 'bundle_version': MANIFEST['bundle_version'], 'actual_backend': REPORT['actual_backend'],
    'gpu_used': REPORT['gpu_used'], 'casmi_infer_source': REPORT['casmi_infer_source'], 'run_id': RUN_ID})
print('saved to', DEST, sorted(p.name for p in DEST.iterdir()))
print('Submitting to Kaggle is MANUAL (download submission.csv from this Drive folder).')